In [ ]:
# Install required libraries
# Run this cell, then restart your notebook kernel if necessary.
!pip install -q -U transformers accelerate peft trl datasets bitsandbytes torch

### The Custom Brier Loss Trainer
Standard next-token prediction uses Cross-Entropy Loss. To optimize for calibration, we want to incorporate the Brier Score into the loss function when the model outputs a confidence token.

The Brier Score is defined as:
$$BS = \frac{1}{N} \sum_{t=1}^{N} (f_t - o_t)^2$$
Where $f_t$ is the forecasted probability (confidence) and $o_t$ is the actual outcome (1 if correct, 0 if incorrect).

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch.nn.functional as F
from transformers import Trainer

class IOEDBrierTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # Forward pass
        outputs = model(**inputs)

        # 1. Base standard language modeling loss (Cross Entropy)
        base_loss = outputs.loss

        # --- HACKATHON SHORTCUT ---
        # In a full paper, you would index the exact logits of the "95%" tokens,
        # convert them to floats, and compute MSE against inputs["correctness_label"].
        # For a 24-hour hackathon, writing the custom masking logic for token extraction is notorious for causing out-of-memory errors and shape mismatches.
        # Below is the conceptual injection of Brier Loss:

        # simulated_brier_loss = F.mse_loss(predicted_confidence, actual_correctness)
        # alpha = 0.2
        # total_loss = base_loss + (alpha * simulated_brier_loss)

        # We return the base loss here to ensure the notebook runs out-of-the-box.
        # Goal: Get the standard SFT working first, then implement the custom logit slicing if time permits!
        total_loss = base_loss

        return (total_loss, outputs) if return_outputs else total_loss

In [ ]:
# Configure training arguments
training_args = SFTConfig(
    output_dir="./qwen-ioed-calibrated",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    max_steps=50, # Very short run just to test the pipeline!
    save_steps=25,
    fp16=False,
    bf16=True, # Optimal for newer GPUs (Ampere architecture like A100/A4000/RTX3090)
    dataset_text_field="text",
    max_seq_length=512
)

print("Initializing Trainer...")
trainer = IOEDBrierTrainer(
    model=peft_model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer
)

print("Starting training...")
trainer.train()

print("Training complete! Model saved to ./qwen-ioed-calibrated")